# Integración agrícola con clima mensual (2003–2025)

Esta notebook integra la producción agrícola con el clima mensual de NASA POWER. Los meses se conservan como características y también se resumen según el ciclo productivo, sin multiplicar los registros agrícolas.

In [1]:
import pandas as pd

from proyecto_agronomia.config import (
    NASA_POWER_PROCESSED_DIR,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
    DGSIAP_DIR
)
from proyecto_agronomia.processing.integracion_climatica import crear_clave_municipal

RUTA_AGRICULTURA = RAW_DATA_DIR / "agricultura"
RUTA_CLIMA_MENSUAL = (
    NASA_POWER_PROCESSED_DIR / "clima_municipal_mensual_2002_2025.csv"
)
RUTA_SALIDA = PROCESSED_DATA_DIR / "agricultura_clima_2003_2025.csv"
pd.set_option("display.max_columns", None)

2026-11-15 18:27:14.629 | INFO     | proyecto_agronomia.config:<module>:14 - PROJ_ROOT path is: /home/fausto/Documents/mcd/1erSemestre/ingenieriaCaracteristicas/proyecto_agronomia


## 1. Cargar la producción agrícola

Como queremos llegar a un dataset que integre los datos de agricultura con los de clima y edafología utilizaremos los datos obtenidos al ejecutar la libreta `agroprod_2`, que son los más limpios y concisos.

In [24]:
archivos_agricultura = sorted(DGSIAP_DIR.glob("ProdAgro*.csv"))
partes_agricultura = []
for archivo in archivos_agricultura:
    anio = 2025 - int(archivo.stem.split("_")[1])
    if 2003 <= anio <= 2025:
        datos = pd.read_csv(archivo, low_memory=False)
        datos["Anio"] = anio
        partes_agricultura.append(datos)
        print(f'{archivo.stem.split("_")[1]}, {anio}: {datos['Precio'].isna().sum()}')

if not partes_agricultura:
    raise FileNotFoundError(f"No se encontraron CSV agrícolas en {RUTA_AGRICULTURA}")

df_agricultura = pd.concat(partes_agricultura, ignore_index=True)
df_agricultura = crear_clave_municipal(df_agricultura)
df_agricultura["Nomcicloproductivo"] = (
    df_agricultura["Nomcicloproductivo"].astype("string").str.strip()
)

periodos_agricolas = {"Otoño-Invierno", "Primavera-Verano", "Perennes"}
periodos_encontrados = set(df_agricultura["Nomcicloproductivo"].dropna().unique())
assert periodos_encontrados <= periodos_agricolas, (
    f"Ciclos agrícolas no contemplados: {sorted(periodos_encontrados - periodos_agricolas)}"
)
print("Agricultura:", df_agricultura.shape)
display(df_agricultura.groupby(["Anio", "Nomcicloproductivo"]).agg(
    registros=("cvegeo", "size"),
    municipios=("cvegeo", "nunique"),
    cultivos=("Idcultivo", "nunique"),
))

0, 2025: 415
10, 2015: 0
11, 2014: 0
12, 2013: 0
13, 2012: 0
14, 2011: 0
15, 2010: 0
16, 2009: 0
17, 2008: 0
18, 2007: 0
19, 2006: 0
1, 2024: 458
20, 2005: 0
21, 2004: 0
22, 2003: 0
2, 2023: 0
3, 2022: 0
4, 2021: 0
5, 2020: 0
6, 2019: 0
7, 2018: 0
8, 2017: 0
9, 2016: 0
Agricultura: (825089, 26)


registros  municipios  cultivos
Anio Nomcicloproductivo                                 
2003 Otoño-Invierno           9858        1897       153
     Perennes                10488        2118       133
     Primavera-Verano        15321        2362       172
2004 Otoño-Invierno           9935        1902       162
     Perennes                10838        2130       132
...                            ...         ...       ...
2024 Perennes                11631        2290       139
     Primavera-Verano        16077        2406       173
2025 Otoño-Invierno           9099        1923       149
     Perennes                11763        2292       137
     Primavera-Verano        15040        2393       167

[69 rows x 3 columns]

In [21]:
df_agricultura.info()

<class 'pandas.DataFrame'>
RangeIndex: 825089 entries, 0 to 825088
Data columns (total 26 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Unnamed: 0          825089 non-null  int64  
 1   Anio                825089 non-null  int64  
 2   Idestado            825089 non-null  int64  
 3   Nomestado           825089 non-null  str    
 4   Idddr               825089 non-null  int64  
 5   Nomddr              825089 non-null  str    
 6   Idcader             825089 non-null  int64  
 7   Nomcader            825089 non-null  str    
 8   Idmunicipio         825089 non-null  int64  
 9   Nommunicipio        825089 non-null  str    
 10  Idciclo             825089 non-null  int64  
 11  Nomcicloproductivo  825089 non-null  string 
 12  Idmodalidad         825089 non-null  int64  
 13  Nommodalidad        825089 non-null  str    
 14  Idunidadmedida      825089 non-null  int64  
 15  Nomunidad           825089 non-null  str    


Todas las entradas están completas a excepción de `Precio`, que tiene valores faltantes en los años 2025 y 2024.

## 2. Cargar y validar el clima mensual

In [7]:
if not RUTA_CLIMA_MENSUAL.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_CLIMA_MENSUAL}. Ejecuta descarga_clima_nasa.ipynb."
    )

columnas_mensuales = [
    "cvegeo", "anio", "mes",
    "temperatura_media_mes_c", "precipitacion_mes_mm",
    "dias_esperados", "dias_temperatura", "dias_precipitacion",
]
df_clima_mensual = pd.read_csv(
    RUTA_CLIMA_MENSUAL,
    usecols=columnas_mensuales,
    dtype={"cvegeo": "string"},
)
df_clima_mensual["cvegeo"] = df_clima_mensual["cvegeo"].str.zfill(5)

clave_mensual = ["cvegeo", "anio", "mes"]
assert not df_clima_mensual.duplicated(clave_mensual).any()
assert set(df_clima_mensual["mes"]) == set(range(1, 13))
assert set(df_clima_mensual["anio"]) == set(range(2002, 2026))
assert df_clima_mensual[[
    "temperatura_media_mes_c", "precipitacion_mes_mm"
]].notna().all().all()
assert (df_clima_mensual["dias_temperatura"] == df_clima_mensual["dias_esperados"]).all()
assert (df_clima_mensual["dias_precipitacion"] == df_clima_mensual["dias_esperados"]).all()
print("Clima mensual:", df_clima_mensual.shape)
display(df_clima_mensual.head())

Clima mensual: (713664, 8)


,cvegeo,anio,mes,temperatura_media_mes_c,precipitacion_mes_mm,dias_esperados,dias_temperatura,dias_precipitacion
0,01001,2002,1,12.089355,17.34,31,31,31
1,01001,2002,2,13.223929,29.07,28,28,28
2,01001,2002,3,16.687097,0.04,31,31,31
3,01001,2002,4,20.270000,2.60,30,30,30
4,01001,2002,5,22.468387,24.57,31,31,31


In [25]:
df_clima_mensual.info()

<class 'pandas.DataFrame'>
RangeIndex: 713664 entries, 0 to 713663
Data columns (total 8 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   cvegeo                   713664 non-null  string 
 1   anio                     713664 non-null  int64  
 2   mes                      713664 non-null  int64  
 3   temperatura_media_mes_c  713664 non-null  float64
 4   precipitacion_mes_mm     713664 non-null  float64
 5   dias_esperados           713664 non-null  int64  
 6   dias_temperatura         713664 non-null  int64  
 7   dias_precipitacion       713664 non-null  int64  
dtypes: float64(2), int64(5), string(1)
memory usage: 43.6 MB


## 3. Construir características mensuales y por ciclo

In [26]:
nombres_meses = {
    1: "ene", 2: "feb", 3: "mar", 4: "abr",
    5: "may", 6: "jun", 7: "jul", 8: "ago",
    9: "sep", 10: "oct", 11: "nov", 12: "dic",
}
indice = ["cvegeo", "anio"]

temperatura_actual = df_clima_mensual.pivot(
    index=indice, columns="mes", values="temperatura_media_mes_c"
).rename(columns={mes: f"temperatura_{nombre}_c" for mes, nombre in nombres_meses.items()})
precipitacion_actual = df_clima_mensual.pivot(
    index=indice, columns="mes", values="precipitacion_mes_mm"
).rename(columns={mes: f"precipitacion_{nombre}_mm" for mes, nombre in nombres_meses.items()})

meses_anteriores = df_clima_mensual.loc[df_clima_mensual["mes"].isin([10, 11, 12])].copy()
meses_anteriores["anio"] += 1
temperatura_anterior = meses_anteriores.pivot(
    index=indice, columns="mes", values="temperatura_media_mes_c"
).rename(columns={mes: f"temperatura_{nombres_meses[mes]}_anterior_c" for mes in [10, 11, 12]})
precipitacion_anterior = meses_anteriores.pivot(
    index=indice, columns="mes", values="precipitacion_mes_mm"
).rename(columns={mes: f"precipitacion_{nombres_meses[mes]}_anterior_mm" for mes in [10, 11, 12]})

df_clima_ancho = (
    temperatura_actual.join(precipitacion_actual, validate="one_to_one")
    .join(temperatura_anterior, validate="one_to_one")
    .join(precipitacion_anterior, validate="one_to_one")
    .reset_index()
)
print("Características mensuales anchas:", df_clima_ancho.shape)

Características mensuales anchas: (59472, 32)


## 4. Resumir por ciclo, integrar y guardar

In [27]:
columnas_base = [
    "cvegeo", "anio", "temperatura_media_mes_c",
    "precipitacion_mes_mm", "dias_esperados",
]

oi_actual = df_clima_mensual.loc[df_clima_mensual["mes"].isin([1, 2, 3]), columnas_base].copy()
oi_actual["periodo_climatico"] = "Otoño-Invierno"
oi_anterior = df_clima_mensual.loc[df_clima_mensual["mes"].isin([10, 11, 12]), columnas_base].copy()
oi_anterior["anio"] += 1
oi_anterior["periodo_climatico"] = "Otoño-Invierno"
pv = df_clima_mensual.loc[df_clima_mensual["mes"].between(4, 9), columnas_base].copy()
pv["periodo_climatico"] = "Primavera-Verano"
perennes = df_clima_mensual[columnas_base].copy()
perennes["periodo_climatico"] = "Perennes"

ventanas = pd.concat([oi_anterior, oi_actual, pv, perennes], ignore_index=True)
ventanas["temperatura_por_dias"] = (
    ventanas["temperatura_media_mes_c"] * ventanas["dias_esperados"]
)
df_clima_ciclos = ventanas.groupby(
    ["cvegeo", "anio", "periodo_climatico"],
    as_index=False,
).agg(
    temperatura_por_dias=("temperatura_por_dias", "sum"),
    precipitacion_ciclo_mm=("precipitacion_mes_mm", "sum"),
    dias_clima_ciclo=("dias_esperados", "sum"),
)
df_clima_ciclos["temperatura_media_ciclo_c"] = (
    df_clima_ciclos.pop("temperatura_por_dias")
    / df_clima_ciclos["dias_clima_ciclo"]
)

df_final = df_agricultura.merge(
    df_clima_ciclos,
    left_on=["cvegeo", "Anio", "Nomcicloproductivo"],
    right_on=["cvegeo", "anio", "periodo_climatico"],
    how="left",
    validate="many_to_one",
).drop(columns=["anio", "periodo_climatico"])
df_final = df_final.merge(
    df_clima_ancho,
    left_on=["cvegeo", "Anio"],
    right_on=["cvegeo", "anio"],
    how="left",
    validate="many_to_one",
).drop(columns="anio")

assert len(df_final) == len(df_agricultura)
columnas_clima = [
    columna for columna in df_final.columns
    if columna.startswith(("temperatura_", "precipitacion_"))
]
assert df_final[columnas_clima].notna().all().all()

resumen_cobertura = df_final.groupby(["Anio", "Nomcicloproductivo"]).agg(
    registros=("cvegeo", "size"),
    municipios=("cvegeo", "nunique"),
    con_temperatura=("temperatura_media_ciclo_c", "count"),
    con_precipitacion=("precipitacion_ciclo_mm", "count"),
)
resumen_cobertura["cobertura_pct"] = (
    100 * resumen_cobertura["con_temperatura"] / resumen_cobertura["registros"]
).round(2)
display(resumen_cobertura)
assert resumen_cobertura["cobertura_pct"].eq(100).all()

RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(RUTA_SALIDA, index=False, encoding="utf-8-sig")
print("Registros agrícolas originales:", len(df_agricultura))
print("Registros integrados:", len(df_final))
print("Dataset guardado:", RUTA_SALIDA)
print("Dimensiones:", df_final.shape)
display(df_final[[
    "Anio", "cvegeo", "Nommunicipio", "Nomcultivo",
    "Nomcicloproductivo", "temperatura_media_ciclo_c",
    "precipitacion_ciclo_mm",
]].head())

registros  municipios  con_temperatura  \
Anio Nomcicloproductivo                                           
2003 Otoño-Invierno           9858        1897             9858   
     Perennes                10488        2118            10488   
     Primavera-Verano        15321        2362            15321   
2004 Otoño-Invierno           9935        1902             9935   
     Perennes                10838        2130            10838   
...                            ...         ...              ...   
2024 Perennes                11631        2290            11631   
     Primavera-Verano        16077        2406            16077   
2025 Otoño-Invierno           9099        1923             9099   
     Perennes                11763        2292            11763   
     Primavera-Verano        15040        2393            15040   

                         con_precipitacion  cobertura_pct  
Anio Nomcicloproductivo                                    
2003 Otoño-Invierno                   9858          100.0  
     Perennes                        10488          100.0  
     Primavera-Verano                15321          100.0  
2004 Otoño-Invierno                   9935          100.0  
     Perennes                        10838          100.0  
...                                    ...            ...  
2024 Perennes                        11631          100.0  
     Primavera-Verano                16077          100.0  
2025 Otoño-Invierno                   9099          100.0  
     Perennes                        11763          100.0  
     Primavera-Verano                15040          100.0  

[69 rows x 5 columns]

Registros agrícolas originales: 825089
Registros integrados: 825089
Dataset guardado: /home/fausto/Documents/mcd/1erSemestre/ingenieriaCaracteristicas/proyecto_agronomia/data/processed/agricultura_clima_2003_2025.csv
Dimensiones: (825089, 59)


,Anio,cvegeo,Nommunicipio,Nomcultivo,Nomcicloproductivo,temperatura_media_ciclo_c,precipitacion_ciclo_mm
0,2025,01001,Aguascalientes,Avena forrajera en verde,Otoño-Invierno,15.777418,21.70
1,2025,01001,Aguascalientes,Brócoli,Otoño-Invierno,15.777418,21.70
2,2025,01001,Aguascalientes,Avena forrajera en verde,Primavera-Verano,20.116776,478.61
3,2025,01001,Aguascalientes,Brócoli,Primavera-Verano,20.116776,478.61
4,2025,01001,Aguascalientes,Calabacita,Primavera-Verano,20.116776,478.61


## 5. Recargar el resultado

In [10]:
df_agricultura_clima = pd.read_csv(
    RUTA_SALIDA,
    dtype={"cvegeo": "string"},
    low_memory=False,
)
df_agricultura_clima["cvegeo"] = df_agricultura_clima["cvegeo"].str.zfill(5)

print("Archivo:", RUTA_SALIDA)
print("Dimensiones:", df_agricultura_clima.shape)
print("Periodo:", df_agricultura_clima["Anio"].min(), "-", df_agricultura_clima["Anio"].max())
display(df_agricultura_clima.head())

Archivo: /home/fausto/Documents/mcd/1erSemestre/ingenieriaCaracteristicas/proyecto_agronomia/data/processed/agricultura_clima_2003_2025.csv
Dimensiones: (825089, 59)
Periodo: 2003 - 2025


,Unnamed: 0,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,Nomcicloproductivo,Idmodalidad,Nommodalidad,Idunidadmedida,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Precio,Valorproduccion,cvegeo,precipitacion_ciclo_mm,dias_clima_ciclo,temperatura_media_ciclo_c,temperatura_ene_c,temperatura_feb_c,temperatura_mar_c,temperatura_abr_c,temperatura_may_c,temperatura_jun_c,temperatura_jul_c,temperatura_ago_c,temperatura_sep_c,temperatura_oct_c,temperatura_nov_c,temperatura_dic_c,precipitacion_ene_mm,precipitacion_feb_mm,precipitacion_mar_mm,precipitacion_abr_mm,precipitacion_may_mm,precipitacion_jun_mm,precipitacion_jul_mm,precipitacion_ago_mm,precipitacion_sep_mm,precipitacion_oct_mm,precipitacion_nov_mm,precipitacion_dic_mm,temperatura_oct_anterior_c,temperatura_nov_anterior_c,temperatura_dic_anterior_c,precipitacion_oct_anterior_mm,precipitacion_nov_anterior_mm,precipitacion_dic_anterior_mm
0,0,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Otoño-Invierno,1,Riego,200201,Tonelada,5490000,Avena forrajera en verde,1500.0,1500.0,0.0,35250.0,23.50,654.00,23053500.00,01001,21.70,182,15.777418,13.361935,16.055357,16.721935,19.179667,22.219355,20.74,19.168387,19.76129,19.605333,18.395484,15.770667,14.339355,8.95,0.0,0.0,0.19,41.41,99.78,139.43,107.24,90.56,6.17,0.0,14.5,17.178387,17.005667,14.407742,10.69,0.11,1.95
1,1,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Otoño-Invierno,1,Riego,200201,Tonelada,5670000,Brócoli,50.0,50.0,0.0,850.0,17.00,6887.78,5854613.00,01001,21.70,182,15.777418,13.361935,16.055357,16.721935,19.179667,22.219355,20.74,19.168387,19.76129,19.605333,18.395484,15.770667,14.339355,8.95,0.0,0.0,0.19,41.41,99.78,139.43,107.24,90.56,6.17,0.0,14.5,17.178387,17.005667,14.407742,10.69,0.11,1.95
2,2,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,Primavera-Verano,1,Riego,200201,Tonelada,5490000,Avena forrajera en verde,165.0,165.0,0.0,3700.0,22.42,645.00,2386500.00,01001,478.61,183,20.116776,13.361935,16.055357,16.721935,19.179667,22.219355,20.74,19.168387,19.76129,19.605333,18.395484,15.770667,14.339355,8.95,0.0,0.0,0.19,41.41,99.78,139.43,107.24,90.56,6.17,0.0,14.5,17.178387,17.005667,14.407742,10.69,0.11,1.95
3,3,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,Primavera-Verano,1,Riego,200201,Tonelada,5670000,Brócoli,60.0,60.0,0.0,785.2,13.09,5161.69,4052958.99,01001,478.61,183,20.116776,13.361935,16.055357,16.721935,19.179667,22.219355,20.74,19.168387,19.76129,19.605333,18.395484,15.770667,14.339355,8.95,0.0,0.0,0.19,41.41,99.78,139.43,107.24,90.56,6.17,0.0,14.5,17.178387,17.005667,14.407742,10.69,0.11,1.95
4,4,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,Primavera-Verano,1,Riego,200201,Tonelada,5740000,Calabacita,30.0,30.0,0.0,490.0,16.33,5285.21,2589752.90,01001,478.61,183,20.116776,13.361935,16.055357,16.721935,19.179667,22.219355,20.74,19.168387,19.76129,19.605333,18.395484,15.770667,14.339355,8.95,0.0,0.0,0.19,41.41,99.78,139.43,107.24,90.56,6.17,0.0,14.5,17.178387,17.005667,14.407742,10.69,0.11,1.95


In [28]:
df_agricultura_clima.info()

<class 'pandas.DataFrame'>
RangeIndex: 825089 entries, 0 to 825088
Data columns (total 59 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Unnamed: 0                     825089 non-null  int64  
 1   Anio                           825089 non-null  int64  
 2   Idestado                       825089 non-null  int64  
 3   Nomestado                      825089 non-null  str    
 4   Idddr                          825089 non-null  int64  
 5   Nomddr                         825089 non-null  str    
 6   Idcader                        825089 non-null  int64  
 7   Nomcader                       825089 non-null  str    
 8   Idmunicipio                    825089 non-null  int64  
 9   Nommunicipio                   825089 non-null  str    
 10  Idciclo                        825089 non-null  int64  
 11  Nomcicloproductivo             825089 non-null  str    
 12  Idmodalidad                    825089 non

## 6. Resumir la edafología por municipio

La capa edafológica se intersecta con los municipios en una proyección métrica. Para cada atributo se conserva la categoría que cubre la mayor superficie; también se calcula el porcentaje del grupo de suelo dominante y el número de grupos presentes. El resumen se guarda para evitar repetir esta operación espacial en cada ejecución.

In [31]:
import geopandas as gpd

from proyecto_agronomia.config import EDAFOLOGIA_RAW_DIR, INEGI_INTERIM_DIR
from proyecto_agronomia.plots import GRUPOS_SUELO_WRB

RUTA_MUNICIPIOS = INEGI_INTERIM_DIR / "municipios_mexico_inegi.geojson"
RUTA_EDAFOLOGIA = (
    EDAFOLOGIA_RAW_DIR / "serie_iii" / "conjunto_de_datos"
    / "conj_nac_inf_edaf_esc_250k_ser_III_area.shp"
)
RUTA_EDAFOLOGIA_MUNICIPAL = PROCESSED_DATA_DIR / "edafologia_municipal.csv"
RECALCULAR_EDAFOLOGIA_MUNICIPAL = False

def resumir_categoria_por_superficie(intersecciones, columna, nombre_salida, areas):
    datos = intersecciones[["cvegeo", columna, "area_interseccion_m2"]].copy()
    datos[columna] = datos[columna].astype("string").str.strip()
    datos = datos.loc[~datos[columna].isin(["", "N", "N/A", "NA"])]
    superficies = (
        datos.groupby(["cvegeo", columna], as_index=False, observed=True)
        ["area_interseccion_m2"].sum()
    )
    dominante = (
        superficies.sort_values(
            ["cvegeo", "area_interseccion_m2", columna],
            ascending=[True, False, True],
        )
        .drop_duplicates("cvegeo")
        .rename(columns={columna: nombre_salida})
    )
    dominante[f"{nombre_salida}_pct"] = (
        dominante["area_interseccion_m2"]
        / dominante["cvegeo"].map(areas) * 100
    ).round(2)
    return dominante.drop(columns="area_interseccion_m2")

if RUTA_EDAFOLOGIA_MUNICIPAL.exists() and not RECALCULAR_EDAFOLOGIA_MUNICIPAL:
    df_edafologia_municipal = pd.read_csv(
        RUTA_EDAFOLOGIA_MUNICIPAL, dtype={"cvegeo": "string"}
    )
else:
    for ruta in [RUTA_MUNICIPIOS, RUTA_EDAFOLOGIA]:
        if not ruta.exists():
            raise FileNotFoundError(f"No se encontró el archivo requerido: {ruta}")

    municipios = gpd.read_file(RUTA_MUNICIPIOS)[["cvegeo", "geometry"]]
    municipios["cvegeo"] = municipios["cvegeo"].astype("string").str.zfill(5)
    municipios = municipios.drop_duplicates("cvegeo").to_crs(epsg=6372)
    edafologia = gpd.read_file(RUTA_EDAFOLOGIA)[
        ["Grupo1", "Clase_tex", "Fase_fis_u", "Fase_qui_u", "geometry"]
    ].to_crs(municipios.crs)

    areas_municipales = municipios.set_index("cvegeo").geometry.area
    intersecciones = gpd.overlay(
        municipios, edafologia, how="intersection", keep_geom_type=False
    )
    intersecciones["area_interseccion_m2"] = intersecciones.geometry.area
    intersecciones = intersecciones.loc[intersecciones["area_interseccion_m2"].gt(0)]

    df_edafologia_municipal = municipios[["cvegeo"]].copy()
    configuracion = {
        "Grupo1": "grupo_suelo_dominante",
        "Clase_tex": "textura_dominante",
        "Fase_fis_u": "fase_fisica_dominante",
        "Fase_qui_u": "fase_quimica_dominante",
    }
    for columna, salida in configuracion.items():
        resumen = resumir_categoria_por_superficie(
            intersecciones, columna, salida, areas_municipales
        )
        df_edafologia_municipal = df_edafologia_municipal.merge(
            resumen, on="cvegeo", how="left", validate="one_to_one"
        )

    diversidad = (
        intersecciones.loc[~intersecciones["Grupo1"].isin(["N", "N/A", "NA"])]
        .groupby("cvegeo")["Grupo1"].nunique()
        .rename("grupos_suelo_n")
        .reset_index()
    )
    df_edafologia_municipal = df_edafologia_municipal.merge(
        diversidad, on="cvegeo", how="left", validate="one_to_one"
    )
    df_edafologia_municipal["nombre_suelo_wrb"] = (
        df_edafologia_municipal["grupo_suelo_dominante"].map(GRUPOS_SUELO_WRB)
    )
    df_edafologia_municipal.to_csv(RUTA_EDAFOLOGIA_MUNICIPAL, index=False)

assert not df_edafologia_municipal["cvegeo"].duplicated().any()
print("Resumen edafológico:", RUTA_EDAFOLOGIA_MUNICIPAL)
print("Municipios resumidos:", len(df_edafologia_municipal))
display(df_edafologia_municipal.head())
print(df_edafologia_municipal.shape)

Resumen edafológico: /home/fausto/Documents/mcd/1erSemestre/ingenieriaCaracteristicas/proyecto_agronomia/data/processed/edafologia_municipal.csv
Municipios resumidos: 2478


,cvegeo,grupo_suelo_dominante,grupo_suelo_dominante_pct,textura_dominante,textura_dominante_pct,fase_fisica_dominante,fase_fisica_dominante_pct,fase_quimica_dominante,fase_quimica_dominante_pct,grupos_suelo_n,nombre_suelo_wrb
0,01001,PH,24.72,2.0,88.81,lep,35.30,NaN,NaN,11,Phaeozem
1,01008,LP,31.44,2.0,95.48,lep,47.28,so,0.49,10,Leptosol
2,01011,DU,41.24,2.0,93.67,pdn,33.27,NaN,NaN,7,Durisol
3,01009,DU,38.81,2.0,77.62,pdn,27.35,NaN,NaN,6,Durisol
4,01007,DU,39.54,2.0,98.41,lep,51.24,so,9.54,11,Durisol


(2478, 11)


## 7. Integrar agricultura, clima y suelo

La unión es `many_to_one`: cada observación agrícola conserva su granularidad y recibe los atributos edafológicos de su municipio.

In [32]:
RUTA_SALIDA_COMPLETA = (
    PROCESSED_DATA_DIR / "agricultura_clima_edafologia_2003_2025.csv"
)

df_agricultura_clima_edafologia = df_agricultura_clima.merge(
    df_edafologia_municipal,
    on="cvegeo",
    how="left",
    validate="many_to_one",
)

assert len(df_agricultura_clima_edafologia) == len(df_agricultura_clima), (
    "La integración edafológica alteró el número de registros agrícolas."
)
cobertura_suelo = (
    df_agricultura_clima_edafologia["grupo_suelo_dominante"].notna().mean() * 100
)
df_agricultura_clima_edafologia.to_csv(RUTA_SALIDA_COMPLETA, index=False)

print("Archivo generado:", RUTA_SALIDA_COMPLETA)
print(f"Registros: {len(df_agricultura_clima_edafologia):,}")
print(f"Cobertura edafológica: {cobertura_suelo:.2f}%")
display(
    df_agricultura_clima_edafologia[[
        "cvegeo", "Anio", "Nomcultivo", "Nomcicloproductivo",
        "temperatura_media_ciclo_c", "precipitacion_ciclo_mm",
        "grupo_suelo_dominante", "nombre_suelo_wrb",
        "grupo_suelo_dominante_pct", "grupos_suelo_n",
        "textura_dominante",
    ]].head()
)
print(df_agricultura_clima_edafologia.info())

Archivo generado: /home/fausto/Documents/mcd/1erSemestre/ingenieriaCaracteristicas/proyecto_agronomia/data/processed/agricultura_clima_edafologia_2003_2025.csv
Registros: 825,089
Cobertura edafológica: 100.00%


,cvegeo,Anio,Nomcultivo,Nomcicloproductivo,temperatura_media_ciclo_c,precipitacion_ciclo_mm,grupo_suelo_dominante,nombre_suelo_wrb,grupo_suelo_dominante_pct,grupos_suelo_n,textura_dominante
0,01001,2025,Avena forrajera en verde,Otoño-Invierno,15.777418,21.70,PH,Phaeozem,24.72,11,2.0
1,01001,2025,Brócoli,Otoño-Invierno,15.777418,21.70,PH,Phaeozem,24.72,11,2.0
2,01001,2025,Avena forrajera en verde,Primavera-Verano,20.116776,478.61,PH,Phaeozem,24.72,11,2.0
3,01001,2025,Brócoli,Primavera-Verano,20.116776,478.61,PH,Phaeozem,24.72,11,2.0
4,01001,2025,Calabacita,Primavera-Verano,20.116776,478.61,PH,Phaeozem,24.72,11,2.0


<class 'pandas.DataFrame'>
RangeIndex: 825089 entries, 0 to 825088
Data columns (total 69 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Unnamed: 0                     825089 non-null  int64  
 1   Anio                           825089 non-null  int64  
 2   Idestado                       825089 non-null  int64  
 3   Nomestado                      825089 non-null  str    
 4   Idddr                          825089 non-null  int64  
 5   Nomddr                         825089 non-null  str    
 6   Idcader                        825089 non-null  int64  
 7   Nomcader                       825089 non-null  str    
 8   Idmunicipio                    825089 non-null  int64  
 9   Nommunicipio                   825089 non-null  str    
 10  Idciclo                        825089 non-null  int64  
 11  Nomcicloproductivo             825089 non-null  str    
 12  Idmodalidad                    825089 non